# 第12–18章网页计算适配验证

教材页码：12章148–156；13章160–163；14章166–175；15章178–182；16章184–191；17章194–204；18章206–212。

输入在notebook构造，计算调用controller→service。全部示例是构造演示，不是假称教材原始数据。先验证每项操作能运行、输入不变、结果可以JSON往返，再用手算/独立关系验证模型与分支。

In [1]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from pandas.testing import assert_frame_equal
from scipy.stats import lognorm

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'service' / 'catalog.py').exists())
sys.path.insert(0, str(root))
from controller.chapter_groups import rates
ops = {op['id']: op for op in rates.operations()}

def run(identifier, parameters=None, frame=None):
    sample, metadata = rates.example_frame(ops[identifier]['example_id'])
    return rates.calculate(sample if frame is None else frame, metadata, {'operation': identifier, 'parameters': parameters or {}})

def metrics(out):
    return {item['name']: item['value'] for item in out['metrics']}

def table(out, index=0):
    return pd.DataFrame(out['tables'][index]['rows'])

results = {}
for identifier, op in ops.items():
    df, metadata = rates.example_frame(op['example_id'])
    before = df.copy(deep=True)
    defaults = {p['name']: p['default'] for p in op['parameters']}
    output = rates.calculate(df, metadata, {'operation': identifier, 'parameters': defaults})
    assert_frame_equal(df, before)
    json.loads(json.dumps(output, allow_nan=False))
    assert output['metrics'] or output['tables']
    results[identifier] = output
assert len(results) == 31
print('31 operations: JSON-safe results, immutable input, runnable default examples')

31 operations: JSON-safe results, immutable input, runnable default examples


## 第12–13章：树、复制、分解与手算值

In [2]:
assert_allclose(metrics(results['ch12_probability'])['向上概率'], .398)
replication = metrics(results['ch12_replication'])
assert_allclose(replication['asset_a'], .5)
assert_allclose(replication['asset_b'], -47.5/1.01)
assert_allclose(replication['price'], .5*98-47.5/1.01)
price = (.2*1000/1.0075 + .8*1000/1.0125)/1.01
assert_allclose(metrics(results['ch12_tree'])['price'], price)
oas = run('ch12_tree', {'mode': 'oas', 'market_price': price})
assert_allclose(metrics(oas)['spread'], 0., atol=1e-10)
calibrated = run('ch12_tree', {'mode': 'calibrate', 'discounts': [1/1.01, price/1000]})
assert_allclose(metrics(calibrated)['price'], price)
assert_allclose(table(calibrated, 1)['0'].iloc[0], .8)
put = run('ch12_tree', {'exercise': 'put', 'exercise_price': 990})
call = run('ch12_tree', {'exercise': 'call', 'exercise_price': 970})
assert metrics(put)['price'] >= price >= metrics(call)['price']
with_cash = run('ch12_tree', {'cashflows': [[2, 2], [0, 2]]})
assert metrics(with_cash)['price'] > price
assert_allclose(table(results['ch12_cmt']).payment, [2500., -2500.])
parts = metrics(results['ch12_attribution'])
assert_allclose(sum(parts[k] for k in ['carry_roll', 'curve', 'spread']), parts['total'])
assert_allclose(parts['total'], 0)
assert_allclose(metrics(results['ch13_discount'])['预期折现因子'], .5/1.025+.5/1.065)
assert_allclose(table(results['ch13_forward']).forward_rate, .04)
decomp = table(results['ch13_decomposition'])
assert_allclose(decomp.forward_rate, .01+decomp.duration*.003-.5*decomp.convexity*.01**2)
assert_allclose(decomp.expected_return, .01+decomp.duration*.001)
print('Ch12–13 hand calculations and tree calibration/OAS/exercise verified')

Ch12–13 hand calculations and tree calibration/OAS/exercise verified


## 第14–15章：漂移、扩散与各离散方法

In [3]:
zeros = pd.DataFrame({'path_a': [0., 0.], 'path_b': [0., 0.]})
g = run('ch14_simulation', {'model': 'gaussian', 'drift_rate': .002, 'dt': .5}, zeros)
assert_allclose(table(g).path_a, [.03, .031, .032])
flat = run('ch14_curve', {'model': 'gaussian', 'drift_rate': 0., 'volatility': 0.})
assert_allclose(table(flat).discount, np.exp(-.03*table(flat).maturity))
ho = run('ch14_curve', {'model': 'holee_curve', 'drift_rate': 0., 'drift_slope': 0.})
assert_allclose(table(ho).discount, np.exp(-.03*table(ho).maturity))
vasicek = table(results['ch14_curve'])
assert_allclose(vasicek['mean'], .05+(.03-.05)*np.exp(-.2*vasicek.maturity))
assert_allclose(vasicek.variance, .01**2*(1-np.exp(-.4*vasicek.maturity))/.4)
log = table(run('ch15_simulation', {'model': 'lognormal'}, zeros))
assert_allclose(log.path_a, .03*np.exp(.002*np.array([0, .5, 1])))
gaussian = table(run('ch15_analytics', {'model': 'gaussian', 'volatility': .01, 'volatility_decay': 0.}))
assert_allclose(gaussian.variance, .0001*gaussian.maturity)
assert_allclose(gaussian['mean'], .03+.002*gaussian.maturity)
assert_allclose(gaussian.discount, np.exp(-.03*gaussian.maturity-.001*gaussian.maturity**2+.0001*gaussian.maturity**3/6))
cir = table(run('ch15_simulation', {'model': 'cir'}, rates.shock_frame(True)))
assert (cir[['path_a', 'path_b']] >= 0).all().all()
bk = table(run('ch15_simulation', {'model': 'bk'}, zeros))
assert_allclose(bk.path_a.iloc[1], np.exp(np.log(.05)+(np.log(.03)-np.log(.05))*np.exp(-.2*.5)))
time_bk = table(run('ch15_simulation', {'model': 'time_bk'}, zeros))
assert_allclose(time_bk.path_a.iloc[1], np.exp(np.log(.03)+.2*(np.log(.05)-np.log(.03))*.5))
assert_allclose(table(results['ch15_tree'])['2'].iloc[1], .03*np.exp(.01))
print('Ch14–15 deterministic paths, Gaussian moments, Ho-Lee curve, CIR positivity and BK transitions verified')

Ch14–15 deterministic paths, Gaussian moments, Ho-Lee curve, CIR positivity and BK transitions verified


## 第16–17章：Gauss+反解、期权平价和密度

In [4]:
state = table(results['ch16_calibration'])
assert_allclose(state[['r', 'm', 'l']], [[.02, .04, .06]], atol=1e-9)
speeds = metrics(run('ch16_calibration', {'mode': 'speeds', 'short_speed': 1., 'medium_speed': .6, 'long_speed': .02}))
assert_allclose([speeds[k] for k in ['short_speed', 'medium_speed', 'long_speed']], [1.0547, .6358, .0165], atol=.001)
vols = metrics(run('ch16_calibration', {'mode': 'volatilities', 'medium_volatility': .01, 'long_volatility': .01, 'correlation': .1}))
assert_allclose([vols[k] for k in ['medium_volatility', 'long_volatility', 'correlation']], [.01092, .00964, .212], atol=.002)
assert_allclose(metrics(run('ch16_calibration', {'mode': 'mean'}))['长期均值μ'], .10555, atol=1e-8)
assert_allclose(list(metrics(results['ch16_hedge']).values()), [-10., -20., -30.])
assert_allclose(metrics(run('ch16_half_life', {'speed': .0165}))['半衰期（年）'], 42.01, atol=.01)
cov = table(results['ch16_curve'], 1)[['r', 'm', 'l']].to_numpy()
assert np.linalg.eigvalsh(cov).min() >= -1e-12
calls = table(results['ch17_options'])
puts = table(run('ch17_options', {'option_sign': 'put'}))
assert_allclose(calls.price-puts.price, 100*np.exp(-.01*calls.maturity)-calls.strike*np.exp(-.03*calls.maturity))
implied = table(run('ch17_options', {'mode': 'implied'}))
assert_allclose(implied.implied_volatility, calls.volatility, atol=1e-9)
assert_allclose(table(results['ch17_surface']).volatility, [.175, .19])
density = table(results['ch17_density'])
assert_allclose(density.density, lognorm.pdf(density.strike, s=.2, scale=100*np.exp(.03-.5*.2**2)), atol=1e-5)
assert_allclose(metrics(results['ch17_jump'])['期权价格'], 4.)
assert_allclose(metrics(results['ch17_delta'])['最小方差Delta'], .48)
print('Ch16–17 calibration recovery, covariance, parity, IV, surface and density verified')

Ch16–17 calibration recovery, covariance, parity, IV, surface and density verified


## 第18章：教材金额量纲与ES组合

In [5]:
assert_allclose(metrics(results['ch18_sensitivity'])['基准敏感度资本'], np.sqrt(20**2+24**2+2*.5*20*24))
assert table(results['ch18_sensitivity']).charge.is_monotonic_increasing
assert_allclose(table(results['ch18_curvature']).curvature_charge, [1., 0.])
assert_allclose(table(results['ch18_default']).default_charge, [2.25])
assert_allclose(metrics(results['ch18_residual'])['残余风险附加'], 3.)
assert_allclose(metrics(results['ch18_liquidity'])['流动性调整ES'], np.sqrt(100+64+72+32+24))
assert_allclose(metrics(results['ch18_stress'])['压力缩放ES'], 30.)
assert_allclose(metrics(results['ch18_weighted'])['加权ES'], 12.)
assert_allclose(metrics(results['ch18_capital'])['资本要求'], 14.5)
assert_allclose(list(metrics(results['ch18_pla']).values()), [0., 0.])
print('Ch18 all nine operations independently anchored')

Ch18 all nine operations independently anchored


## 全部分支与边界：不隐式清洗，不执行任意表达式

In [6]:
branches = []
for identifier, op in ops.items():
    for parameter in op['parameters']:
        if parameter['type'] != 'select':
            continue
        for option in parameter['options']:
            changes = {parameter['name']: option['value']}
            frame = rates.shock_frame(True) if identifier == 'ch15_simulation' and changes.get('model') == 'cir' else None
            output = run(identifier, changes, frame)
            json.dumps(output, allow_nan=False)
            branches.append((identifier, parameter['name'], option['value']))
assert len(branches) == 34

def rejects(identifier, parameters=None, frame=None):
    try:
        run(identifier, parameters, frame)
    except (ValueError, KeyError):
        return
    raise AssertionError(f'{identifier} failed to reject invalid input')

rejects('ch12_tree', {'probabilities': [[2., .5], [None, .5]]})
rejects('ch12_tree', {'rates': [[.02]]})
rejects('ch12_replication', frame=pd.DataFrame({'asset_a': [1., 1.], 'asset_b': [1., 1.], 'target': [1., 1.]}))
rejects('ch13_discount', frame=pd.DataFrame({'rate': [.03, .04], 'probability': [.8, .8]}))
rejects('ch14_simulation', frame=pd.DataFrame({'path_a': [np.nan], 'path_b': [0.]}))
rejects('ch15_simulation', {'model': 'cir'})
rejects('ch15_tree', {'n_steps': 3.5})
rejects('ch16_curve', {'reference_maturities': [2., 2.]})
rejects('ch17_surface', frame=pd.DataFrame({'maturity': [5.], 'moneyness': [1.]}))
rejects('ch17_density', frame=pd.DataFrame({'strike': [100., 90., 110.], 'call_price': [8., 15., 4.]}))
rejects('ch18_sensitivity', {'correlation': [[1., 2.], [2., 1.]]})
rejects('ch18_liquidity', frame=pd.DataFrame({'es': [10., 8.], 'horizon': [20., 10.]}))
rejects('ch18_pla', frame=pd.DataFrame({'actual': [1., 1.], 'model': [0., 0.]}))
rejects('ch18_stress', {'current_reduced': 0.})
rejects('ch18_capital', {'unexpected': 'arbitrary_expression'})
# Generic column mapping survives renaming; no source field names are hardcoded in calculations.
renamed = pd.DataFrame({'cash': [100., 200.], 'factor': [.01, .01]})
assert_allclose(metrics(run('ch18_residual', {'notional_column': 'cash', 'weight_column': 'factor'}, renamed))['残余风险附加'], 3.)
print(f'{len(branches)} selector branches and 15 invalid-boundary cases verified')

34 selector branches and 15 invalid-boundary cases verified
